# EDA — Workshop 2: Spotify + Grammy Awards
**Goal**: understand the structure, quality, and distribution of both datasets
to guide cleaning and transformation decisions in the ETL pipeline.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)

SPOTIFY_PATH = '../data/spotify_dataset.csv'
GRAMMYS_PATH = '../data/the_grammy_awards.csv'

---
## 1. Data loading

In [ ]:
spotify  = pd.read_csv(SPOTIFY_PATH, index_col=0)
grammys  = pd.read_csv(GRAMMYS_PATH)

print(f'Spotify : {spotify.shape[0]:,} rows × {spotify.shape[1]} columns')
print(f'Grammys : {grammys.shape[0]:,} rows × {grammys.shape[1]} columns')

---
## 2. Spotify dataset
### 2.1 General structure

In [ ]:
spotify.info()

In [ ]:
spotify.describe().round(3)

### 2.2 Null values

In [ ]:
nulls = spotify.isnull().sum()
nulls = nulls[nulls > 0]
if nulls.empty:
    print('No null values in Spotify')
else:
    print(nulls)

### 2.3 Duplicates on track_id

In [ ]:
total_rows    = len(spotify)
unique_tracks = spotify['track_id'].nunique()
duplicates    = total_rows - unique_tracks

print(f'Total rows           : {total_rows:,}')
print(f'Unique track_ids     : {unique_tracks:,}')
print(f'Duplicate track_ids  : {duplicates:,}')
print()
print('Note: the same track_id can appear under multiple genres.')
print('This is handled in cleaning with drop_duplicates(subset=["track_id"]).')

# Example of a duplicated track
dup_id = spotify[spotify['track_id'].duplicated(keep=False)]['track_id'].iloc[0]
spotify[spotify['track_id'] == dup_id][['track_id', 'track_name', 'artists', 'track_genre']]

### 2.4 Popularity distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(spotify['popularity'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Popularity distribution (Spotify)')
axes[0].set_xlabel('Popularity (0–100)')
axes[0].set_ylabel('Frequency')

# Boxplot
axes[1].boxplot(spotify['popularity'], vert=False, patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.7))
axes[1].set_title('Popularity boxplot')
axes[1].set_xlabel('Popularity (0–100)')

plt.tight_layout()
plt.show()

print(f"Tracks with popularity = 0: {(spotify['popularity'] == 0).sum():,} ({(spotify['popularity'] == 0).mean():.1%})")

### 2.5 Music genres

In [ ]:
print(f"Unique genres: {spotify['track_genre'].nunique()}")

top_genres = spotify['track_genre'].value_counts().head(15)
top_genres.plot(kind='barh', color='teal', figsize=(12, 6))
plt.title('Top 15 genres by number of tracks')
plt.xlabel('Number of tracks')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

### 2.6 Average popularity by genre

In [ ]:
pop_by_genre = (
    spotify.groupby('track_genre')['popularity']
    .mean()
    .sort_values(ascending=False)
    .head(15)
)

pop_by_genre.plot(kind='bar', color='coral', figsize=(14, 5))
plt.title('Top 15 genres by average popularity')
plt.ylabel('Average popularity')
plt.xlabel('Genre')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### 2.7 Correlations between audio features

In [ ]:
audio_features = [
    'popularity', 'danceability', 'energy', 'loudness',
    'speechiness', 'acousticness', 'instrumentalness',
    'liveness', 'valence', 'tempo'
]

corr = spotify[audio_features].corr()

plt.figure(figsize=(12, 9))
mask = pd.DataFrame(False, index=corr.index, columns=corr.columns)
# Show lower triangle only
for i in range(len(corr)):
    for j in range(i):
        mask.iloc[i, j] = False
    for j in range(i, len(corr)):
        mask.iloc[i, j] = True

sns.heatmap(
    corr,
    annot=True,
    fmt='.2f',
    cmap='RdYlGn',
    center=0,
    linewidths=0.5,
    vmin=-1,
    vmax=1,
)
plt.title('Correlation between audio features (Spotify)')
plt.tight_layout()
plt.show()

# Highest correlations with popularity
print('Correlation with popularity:')
print(corr['popularity'].drop('popularity').sort_values(ascending=False))

### 2.8 Artist string format (separator `;`)

In [ ]:
# Multiple artists use ';' as separator
multi_artist = spotify[spotify['artists'].str.contains(';', na=False)]
print(f"Tracks with multiple artists: {len(multi_artist):,} ({len(multi_artist)/len(spotify):.1%})")
print()
print('Examples:')
multi_artist[['track_name', 'artists']].head(5)

---
## 3. Grammy Awards dataset
### 3.1 General structure

In [ ]:
grammys.info()

In [ ]:
grammys.head(3)

### 3.2 Key finding: winners only

In [ ]:
# IMPORTANT: check distribution of the 'winner' column
print('Distribution of column winner:')
print(grammys['winner'].value_counts(dropna=False))
print()
print('>>> CONCLUSION: The dataset only contains WINNERS (winner=True).')
print('    There are no nominees who did not win. The analysis will compare:')
print('    - Grammy-winning artists (in this dataset)')
print('    - Artists without a Grammy (on Spotify but not in Grammys)')

### 3.3 Year range and unique artists

In [ ]:
print(f"Year range     : {grammys['year'].min()} – {grammys['year'].max()}")
print(f"Ceremonies     : {grammys['year'].nunique()}")
print(f"Unique artists : {grammys['artist'].nunique():,}")
print(f"Categories     : {grammys['category'].nunique():,}")

### 3.4 Null values in Grammys

In [ ]:
nulls_g = grammys.isnull().sum()
print(nulls_g[nulls_g > 0])
print()
print('ETL decision: null artist → use nominee as fallback (applied in clean_grammys)')

### 3.5 Awards per year

In [ ]:
awards_by_year = grammys.groupby('year').size().reset_index(name='count')

plt.figure(figsize=(16, 5))
plt.plot(awards_by_year['year'], awards_by_year['count'], color='goldenrod', linewidth=2, marker='o', markersize=3)
plt.fill_between(awards_by_year['year'], awards_by_year['count'], alpha=0.2, color='goldenrod')
plt.title('Grammy Awards given per year')
plt.xlabel('Year')
plt.ylabel('Awards')
plt.tight_layout()
plt.show()

### 3.6 Top categories and most awarded artists

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Top categories
top_cat = grammys['category'].value_counts().head(15)
axes[0].barh(top_cat.index, top_cat.values, color='sandybrown')
axes[0].set_title('Top 15 Grammy categories')
axes[0].set_xlabel('Awards given')
axes[0].invert_yaxis()

# Top artists by number of wins
top_artists = grammys['artist'].value_counts().head(15)
axes[1].barh(top_artists.index, top_artists.values, color='steelblue')
axes[1].set_title('Top 15 artists by Grammy wins')
axes[1].set_xlabel('Awards won')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

---
## 4. Cross-dataset analysis: matching

In [ ]:
import re

def normalize(name: str) -> str:
    if not isinstance(name, str):
        return ''
    name = name.strip().lower()
    name = re.sub(r"['\"-]", '', name)
    name = re.sub(r'\s+', ' ', name)
    return name

# Primary Spotify artist (first name when there are several)
spotify['artist_primary'] = spotify['artists'].str.split(';').str[0].str.strip()
spotify['artist_key']     = spotify['artist_primary'].map(normalize)

grammys['artist_key'] = grammys['artist'].map(normalize)

# Unique artists in each dataset
spotify_artists  = set(spotify['artist_key'].dropna().unique())
grammy_artists   = set(grammys['artist_key'].dropna().unique())

match = spotify_artists & grammy_artists
match_rate = len(match) / len(grammy_artists)

print(f'Unique artists in Spotify   : {len(spotify_artists):,}')
print(f'Unique artists in Grammys   : {len(grammy_artists):,}')
print(f'Artists in common (match)   : {len(match):,}')
print(f'% of Grammy artists in Spotify: {match_rate:.1%}')

In [ ]:
# Check: some Grammy winners that ARE on Spotify
print('Grammy winners WITH presence on Spotify:')
print(sorted(list(match))[:20])

In [ ]:
# Grammy winners NOT on Spotify (classic / legacy artists, etc.)
only_grammy = grammy_artists - spotify_artists
print(f'Grammy winners WITHOUT presence on Spotify: {len(only_grammy):,}')
print('Examples:', sorted(list(only_grammy))[:10])

---
## 5. Conclusions for the ETL pipeline

### Main findings

| # | Finding | ETL action |
|---|---------|------------|
| 1 | **Spotify has 24,259 duplicate track_ids** (same track under multiple genres) | `drop_duplicates(subset=['track_id'])` in `clean_spotify` |
| 2 | **Grammy dataset contains winners only** (no non-winning nominees) | Analysis compares Grammy winners vs artists without a Grammy on Spotify |
| 3 | **Spotify artists use `;` as separator** for collaborations | Take first artist: `artists.split(';')[0]` |
| 4 | **Grammys has null artists** (instrumental/technical categories) | Fallback: use `nominee` when `artist` is null |
| 5 | **~X% of Grammy winners appear on Spotify** | The merge will yield many NaN rows for artists with no match |
| 6 | **Many popularity values = 0** (obscure / low-stream tracks) | Consider filtering `popularity > 0` in visualizations |
| 7 | **Energy and loudness are strongly positively correlated** | Do not drop — both capture different information |
| 8 | **Acousticness and energy correlate negatively** | Expected: acoustic tracks tend to have lower electric energy |

### Transformation decisions
- **Spotify cleaning**: dedupe by `track_id`, normalize primary artist
- **Grammys cleaning**: normalize `artist_key`, artist → nominee fallback
- **Merge**: LEFT JOIN Spotify ← Grammys on `artist_key`
- **Star schema**: grain = (track, artist, year, award category)